In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import pickle
from tqdm import tqdm

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
mg_mapping = pd.read_csv('SATURN_mapping/CJ_30_seeds.csv', index_col = 'barcode')

In [6]:
mo_mapping = pd.read_csv('SATURN_mapping/vole_CJ_30_seeds.csv', index_col = 'barcode')

In [7]:
sam = SAM()
sam.load_data('Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad')

In [8]:
mg_mapping.columns

Index(['seed_0', 'seed_1', 'seed_2', 'seed_3', 'seed_4', 'seed_5', 'seed_6',
       'seed_7', 'seed_8', 'seed_9', 'seed_10', 'seed_11', 'seed_12',
       'seed_13', 'seed_14', 'seed_15', 'seed_16', 'seed_17', 'seed_18',
       'seed_19', 'seed_20', 'seed_21', 'seed_22', 'seed_23', 'seed_24',
       'seed_25', 'seed_26', 'seed_27', 'seed_28', 'seed_29'],
      dtype='object')

In [9]:
sam.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [10]:
level = 'eq_subclass_lc'

In [11]:
cj_clusters = sam.adata.obs[level].to_frame()
cj_clusters.colums = [level]

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  


In [12]:
cj_clusters

,eq_subclass_lc
AAACCCAAGACGACTG Run 11 sample 8,170
AAACCCAAGTGCTCAT Run 11 sample 8,81
AAACCCAGTCGAAACG Run 11 sample 8,23
AAACCCAGTCTTGAAC Run 11 sample 8,292
AAACCCAGTGCCGGTT Run 11 sample 8,84
...,...
TTTGGTTTCCACAGCG Run 11 sample 7,135
TTTGGTTTCCATCAGA Run 11 sample 7,72
TTTGGTTTCGGTGCAC Run 11 sample 7,174
TTTGTTGAGGAATTAC Run 11 sample 7,67


In [14]:
cj_clusters[level].unique()

array([170,  81,  23, 292,  84,  83, 142, 147, 139, 134,   0,   6,  29,
       281, 172, 191, 185,  75, 240, 154, 316, 102,  45, 199, 124, 282,
       297, 123, 146,   8,  22, 248,  30,  56,  87, 269, 188, 161, 211,
       363,  66,  96, 189, 230, 153,  61, 294,  77, 137,  67, 227, 302,
        46, 251, 117,  17, 202, 184,  85,  39,   9, 138, 198, 157, 206,
       338,  88,  27, 296,  68, 234,  78, 178, 175,  89, 260, 356, 340,
       318,  40,  63, 106, 255,  62, 239, 203, 268,  70, 263,  52,  71,
       380, 180,  99,  64, 256,  12,  28, 128, 317,  15, 309, 136, 173,
       183,  92, 214, 187, 122,  34, 242, 197, 158,  38, 308,  42, 367,
       166,  41, 141, 225, 111, 270, 119, 140, 289, 176, 220, 105, 293,
       244, 165, 164, 208, 162, 104, 192, 366, 216, 232, 101,   4, 228,
       125, 217,  21,  54, 221,  51, 307, 115, 279, 155, 403, 320, 159,
       148, 112, 286, 339, 114, 132, 327, 174, 349,   3, 283, 328, 103,
       235,  65, 224,  13,  82, 222, 233, 108, 150,  57, 169, 39

In [15]:
df_comb_mapping = pd.DataFrame(index = cj_clusters.index, columns = ['seed_' + str(i) for i in range(30)])

In [16]:
cj_clusters

,eq_subclass_lc
AAACCCAAGACGACTG Run 11 sample 8,170
AAACCCAAGTGCTCAT Run 11 sample 8,81
AAACCCAGTCGAAACG Run 11 sample 8,23
AAACCCAGTCTTGAAC Run 11 sample 8,292
AAACCCAGTGCCGGTT Run 11 sample 8,84
...,...
TTTGGTTTCCACAGCG Run 11 sample 7,135
TTTGGTTTCCATCAGA Run 11 sample 7,72
TTTGGTTTCGGTGCAC Run 11 sample 7,174
TTTGTTGAGGAATTAC Run 11 sample 7,67


In [17]:
for i in tqdm(range(30)):
    mapping_dict = {}
    for lc in range(cj_clusters[level].nunique()):
        mg_mapping_set = mg_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        mo_mapping_set = mo_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        if mg_mapping_set.mode()[0] == mo_mapping_set.mode()[0] and len(mg_mapping_set) >25:
            mapping_dict[lc] = mg_mapping_set.mode()[0] 
        else:
            mapping_dict[lc] = 'Unlabeled'
    new_mapping = [mapping_dict[item] for item in cj_clusters[level]]
    df_comb_mapping.loc[:,'seed_' + str(i)] = new_mapping

100%|███████████████████████████████████████████| 30/30 [00:20<00:00,  1.49it/s]


In [82]:
df_comb_mapping.to_csv('SATURN_mapping/MV_mm_mo_SATURN_30_seed.csv')